In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
from tqdm import tqdm


In [3]:
import pandas as pd

df = pd.read_csv("HAM10000_metadata.csv")

# Binary label
df['label'] = df['dx'].apply(lambda x: 1 if x == 'mel' else 0)

print(df['label'].value_counts())


label
0    8902
1    1113
Name: count, dtype: int64


In [4]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df['label'],
    random_state=42
)

print("Train:", train_df['label'].value_counts())
print("Val:", val_df['label'].value_counts())
print("Test:", test_df['label'].value_counts())


Train: label
0    6231
1     779
Name: count, dtype: int64
Val: label
0    1335
1     167
Name: count, dtype: int64
Test: label
0    1336
1     167
Name: count, dtype: int64


In [5]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [6]:
from torch.utils.data import Dataset
from PIL import Image
import os

class HAM10000Dataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'image_id']
        label = self.df.loc[idx, 'label']
        img_path = os.path.join(self.image_dir, img_id + ".jpg")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [7]:
from torch.utils.data import DataLoader

image_dir = "HAM10000_images"

train_dataset = HAM10000Dataset(train_df, image_dir, train_transform)
val_dataset   = HAM10000Dataset(val_df, image_dir, val_test_transform)
test_dataset  = HAM10000Dataset(test_df, image_dir, val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [8]:
images, labels = next(iter(train_loader))
print(images.shape)
print(labels)


torch.Size([32, 3, 224, 224])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0])


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [11]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)


In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [13]:
num_epochs = 10

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------- TRAIN --------
    model.train()
    running_train_loss = 0.0

    train_pbar = tqdm(train_loader, desc="Training", leave=False)

    for images, labels in train_pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        # Update progress bar
        train_pbar.set_postfix(loss=loss.item())

    avg_train_loss = running_train_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    val_pbar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, labels in val_pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Update progress bar
            val_pbar.set_postfix(loss=loss.item())

    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.4f}")



Epoch 1/10


Train Loss: 0.2554 | Val Loss: 0.2100 | Val Acc: 0.9088

Epoch 2/10


Train Loss: 0.2127 | Val Loss: 0.2188 | Val Acc: 0.9081

Epoch 3/10


Train Loss: 0.1767 | Val Loss: 0.2378 | Val Acc: 0.9008

Epoch 4/10


Train Loss: 0.1512 | Val Loss: 0.2160 | Val Acc: 0.9101

Epoch 5/10


...

Train Loss: 0.1008 | Val Loss: 0.2228 | Val Acc: 0.9161

Epoch 7/10


Train Loss: 0.0961 | Val Loss: 0.3629 | Val Acc: 0.9028

Epoch 8/10


Train Loss: 0.0988 | Val Loss: 0.2637 | Val Acc: 0.9228

Epoch 9/10


Train Loss: 0.0878 | Val Loss: 0.2276 | Val Acc: 0.9288

Epoch 10/10


Train Loss: 0.0754 | Val Loss: 0.2275 | Val Acc: 0.9274


In [33]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["non-mel", "mel"]))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


Classification Report:
              precision    recall  f1-score   support

     non-mel       0.89      0.18      0.31      1336
         mel       0.11      0.81      0.20       167

    accuracy                           0.25      1503
   macro avg       0.50      0.50      0.25      1503
weighted avg       0.80      0.25      0.29      1503

Confusion Matrix:
[[ 246 1090]
 [  31  136]]


In [34]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()


In [35]:
criterion = FocalLoss(alpha=0.25, gamma=2.0)


In [36]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)


In [37]:
from tqdm import tqdm


In [38]:
num_epochs = 10

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------- TRAIN --------
    model.train()
    running_train_loss = 0.0

    train_pbar = tqdm(train_loader, desc="Training", leave=False)

    for images, labels in train_pbar:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

        # Update progress bar
        train_pbar.set_postfix(loss=loss.item())

    avg_train_loss = running_train_loss / len(train_loader)

    # -------- VALIDATION --------
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    val_pbar = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, labels in val_pbar:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Update progress bar
            val_pbar.set_postfix(loss=loss.item())

    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.4f}")



Epoch 1/10


Train Loss: 0.1758 | Val Loss: 0.1763 | Val Acc: 0.1258

Epoch 2/10


Train Loss: 0.1765 | Val Loss: 0.1848 | Val Acc: 0.1218

Epoch 3/10


Train Loss: 0.1762 | Val Loss: 0.1878 | Val Acc: 0.1212

Epoch 4/10


Train Loss: 0.1761 | Val Loss: 0.1802 | Val Acc: 0.1292

Epoch 5/10


...

Train Loss: 0.1767 | Val Loss: 0.1766 | Val Acc: 0.1265

Epoch 7/10


Train Loss: 0.1764 | Val Loss: 0.1770 | Val Acc: 0.1278

Epoch 8/10


Train Loss: 0.1761 | Val Loss: 0.1809 | Val Acc: 0.1272

Epoch 9/10


Train Loss: 0.1763 | Val Loss: 0.1988 | Val Acc: 0.1198

Epoch 10/10


Train Loss: 0.1760 | Val Loss: 0.2048 | Val Acc: 0.1185


In [39]:
mel_train_df = train_df[train_df['label'] == 1].reset_index(drop=True)
print("Real melanoma samples:", len(mel_train_df))


Real melanoma samples: 779


In [40]:
from torchvision import transforms

gan_transform = transforms.Compose([
    transforms.Resize(64),
    transforms.CenterCrop(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [41]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os

class MelanomaGANDataset(Dataset):
    def __init__(self, df, image_dir, transform):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'image_id']
        img_path = os.path.join(self.image_dir, img_id + ".jpg")
        img = Image.open(img_path).convert("RGB")
        return self.transform(img)


In [42]:
gan_dataset = MelanomaGANDataset(
    mel_train_df,
    image_dir="HAM10000_images",
    transform=gan_transform
)

gan_loader = DataLoader(
    gan_dataset,
    batch_size=64,
    shuffle=True
)


In [43]:
import torch
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, nz=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(nz, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 3, 4, 2, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)


In [47]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
        )

        # THIS is the key fix
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # → (B, 512, 1, 1)
            nn.Conv2d(512, 1, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x.view(-1)


In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G = Generator().to(device)
D = Discriminator().to(device)

criterion_gan = nn.BCELoss()

opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

nz = 100
gan_epochs = 30


In [49]:
for epoch in range(gan_epochs):
    for real_imgs in gan_loader:
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)

        real_labels = torch.ones(batch_size, device=device)
        fake_labels = torch.zeros(batch_size, device=device)

        # ---- Train Discriminator ----
        noise = torch.randn(batch_size, nz, 1, 1, device=device)
        fake_imgs = G(noise)

        loss_D = (
            criterion_gan(D(real_imgs), real_labels) +
            criterion_gan(D(fake_imgs.detach()), fake_labels)
        )

        opt_D.zero_grad()
        loss_D.backward()
        opt_D.step()

        # ---- Train Generator ----
        loss_G = criterion_gan(D(fake_imgs), real_labels)

        opt_G.zero_grad()
        loss_G.backward()
        opt_G.step()

    print(f"Epoch [{epoch+1}/{gan_epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")


Epoch [1/30] | D Loss: 1.1299 | G Loss: 0.8504
Epoch [2/30] | D Loss: 1.0387 | G Loss: 1.0165
Epoch [3/30] | D Loss: 0.9918 | G Loss: 1.1441
Epoch [4/30] | D Loss: 0.9413 | G Loss: 1.1196
Epoch [5/30] | D Loss: 1.1780 | G Loss: 1.0133
Epoch [6/30] | D Loss: 1.0661 | G Loss: 1.1456
Epoch [7/30] | D Loss: 1.0906 | G Loss: 1.2574
Epoch [8/30] | D Loss: 1.2024 | G Loss: 1.2392
Epoch [9/30] | D Loss: 0.8516 | G Loss: 1.2457
Epoch [10/30] | D Loss: 1.0202 | G Loss: 1.3213
Epoch [11/30] | D Loss: 1.0068 | G Loss: 1.3725
Epoch [12/30] | D Loss: 1.0374 | G Loss: 1.3418
Epoch [13/30] | D Loss: 0.6387 | G Loss: 1.4653
Epoch [14/30] | D Loss: 0.7087 | G Loss: 1.5654
Epoch [15/30] | D Loss: 0.8366 | G Loss: 2.0079
Epoch [16/30] | D Loss: 0.8633 | G Loss: 1.3953
Epoch [17/30] | D Loss: 1.0504 | G Loss: 1.4527
Epoch [18/30] | D Loss: 0.7679 | G Loss: 1.4490
Epoch [19/30] | D Loss: 0.4922 | G Loss: 1.6967
Epoch [20/30] | D Loss: 0.6465 | G Loss: 1.4624
Epoch [21/30] | D Loss: 0.5003 | G Loss: 1.7872
E

In [52]:
G.eval()

num_synthetic = 1000  # you can tune this
noise = torch.randn(num_synthetic, nz, 1, 1, device=device)

with torch.no_grad():
    synthetic_imgs = G(noise).cpu()


In [53]:
import os

synthetic_dir = "synthetic_mel"
os.makedirs(synthetic_dir, exist_ok=True)


In [54]:
from torchvision.utils import save_image

for i in range(num_synthetic):
    save_image(
        synthetic_imgs[i],
        os.path.join(synthetic_dir, f"mel_gan_{i}.png"),
        normalize=True
    )

print(f"{num_synthetic} synthetic melanoma images saved to '{synthetic_dir}/'")


1000 synthetic melanoma images saved to 'synthetic_mel/'


In [55]:
import pandas as pd

gan_df = pd.DataFrame({
    "image_id": [f"mel_gan_{i}" for i in range(num_synthetic)],
    "label": [1] * num_synthetic  # 1 = melanoma
})


In [56]:
train_df_gan = pd.concat([train_df, gan_df], ignore_index=True)

print("Original train size:", len(train_df))
print("Train size after GAN:", len(train_df_gan))


Original train size: 7010
Train size after GAN: 8010


In [68]:
from torch.utils.data import Dataset
from PIL import Image
import os

class HAM10000WithGAN(Dataset):
    def __init__(self, df, real_dir, gan_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.real_dir = real_dir
        self.gan_dir = gan_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, "image_id"]
        label = self.df.loc[idx, "label"]

        # Decide image source
        if img_id.startswith("mel_gan"):
            img_path = os.path.join(self.gan_dir, img_id + ".png")
        else:
            img_path = os.path.join(self.real_dir, img_id + ".jpg")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [69]:
class HAM10000WithGAN(Dataset):
    def __init__(self, df, real_dir, gan_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.real_dir = real_dir
        self.gan_dir = gan_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, "image_id"]
        label = self.df.loc[idx, "label"]

        # Decide source folder
        if img_id.startswith("mel_gan"):
            img_path = os.path.join(self.gan_dir, img_id + ".png")
        else:
            img_path = os.path.join(self.real_dir, img_id + ".jpg")

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [70]:
train_dataset_gan = HAM10000WithGAN(
    train_df_gan,
    real_dir="HAM10000_images",
    gan_dir="synthetic_mel",
    transform=train_transform
)

train_loader_gan = DataLoader(
    train_dataset_gan,
    batch_size=32,
    shuffle=True
)


In [71]:
from torchvision.models import resnet18, ResNet18_Weights
import torch.nn as nn

model = resnet18(weights=ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)


In [72]:
criterion = FocalLoss(alpha=0.25, gamma=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


In [74]:
num_epochs = 10

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")

    # -------- TRAIN --------
    model.train()
    running_train_loss = 0.0

    for images, labels in train_loader_gan:   # 👈 GAN loader
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader_gan)

    # -------- VALIDATION --------
    model.eval()
    running_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = running_val_loss / len(val_loader)
    val_accuracy = correct / total

    print(f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_accuracy:.4f}")



Epoch 1/10
Train Loss: 0.0137 | Val Loss: 0.0137 | Val Acc: 0.9154

Epoch 2/10
Train Loss: 0.0114 | Val Loss: 0.0151 | Val Acc: 0.9048

Epoch 3/10
Train Loss: 0.0102 | Val Loss: 0.0135 | Val Acc: 0.9115

Epoch 4/10
Train Loss: 0.0091 | Val Loss: 0.0131 | Val Acc: 0.9194

Epoch 5/10
Train Loss: 0.0079 | Val Loss: 0.0129 | Val Acc: 0.9241

Epoch 6/10
Train Loss: 0.0070 | Val Loss: 0.0163 | Val Acc: 0.9168

Epoch 7/10
Train Loss: 0.0062 | Val Loss: 0.0159 | Val Acc: 0.9248

Epoch 8/10
Train Loss: 0.0056 | Val Loss: 0.0162 | Val Acc: 0.9161

Epoch 9/10
Train Loss: 0.0049 | Val Loss: 0.0171 | Val Acc: 0.9088

Epoch 10/10
Train Loss: 0.0048 | Val Loss: 0.0191 | Val Acc: 0.9181


In [75]:
from sklearn.metrics import classification_report, confusion_matrix

model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["non-mel", "mel"]))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))


Classification Report:
              precision    recall  f1-score   support

     non-mel       0.94      0.97      0.96      1336
         mel       0.70      0.53      0.61       167

    accuracy                           0.92      1503
   macro avg       0.82      0.75      0.78      1503
weighted avg       0.92      0.92      0.92      1503

Confusion Matrix:
[[1298   38]
 [  78   89]]


In [1]:
import pandas as pd

# Load ISIC ground truth
isic_gt = pd.read_csv("C:\Users\yadav\Downloads\isic\ISIC_2019_Training_GroundTruth.csv")

# The melanoma column is usually named 'MEL'
# Create binary label
isic_gt["label"] = isic_gt["MEL"]

# Keep only what we need
isic_df = isic_gt[["image", "label"]]

# Rename to match HAM10000
isic_df = isic_df.rename(columns={"image": "image_id"})


SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (3531690262.py, line 4)